## Setup & Installation

In [15]:
# Install required libraries
!pip install langchain langchain-core --quiet
!pip install langchain langchain-core

In [16]:
# Import all necessary modules from LangChain (compatible with newer versions)
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

## Task 1: Replace Hardcoded Prompts

In [17]:
# Original (BAD): def explain_topic(topic): return f"Explain {topic} in simple terms for beginners"
# Requirement: No f-strings, use PromptTemplate, must be reusable

# Define a reusable PromptTemplate with input_variables
explain_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Reusable function using the template (no f-strings, no hardcoding)
def explain_topic(topic):
    """Generate a beginner-friendly explanation prompt for any topic."""
    return explain_template.format(topic=topic)

# Test the template with different topics
print("=== Task 1: Replace Hardcoded Prompts ===")
print(explain_topic("Machine Learning"))
print(explain_topic("Neural Networks"))
print(explain_topic("Python"))

=== Task 1: Replace Hardcoded Prompts ===
Explain Machine Learning in simple terms for beginners
Explain Neural Networks in simple terms for beginners
Explain Python in simple terms for beginners



## Task 2: Multi-Input Prompt System



In [18]:
# Template: "Explain {topic} for {audience} in a {tone} tone"
# Test Cases: AI|beginners|friendly, Python|kids|fun, Deep Learning|engineers|technical

# Define multi-input PromptTemplate
multi_input_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Reusable function to generate multi-input prompts
def explain_for_audience(topic, audience, tone):
    """Generate a prompt tailored to a specific audience and tone."""
    return multi_input_template.format(topic=topic, audience=audience, tone=tone)

# Define all test cases as a list of dictionaries (no hardcoding)
test_cases = [
    {"topic": "AI",            "audience": "beginners",  "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",        "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers",  "tone": "technical"},
]

print("=== Task 2: Multi-Input Prompt System ===")
for case in test_cases:
    prompt = explain_for_audience(**case)
    print(f"Output: {prompt}")

=== Task 2: Multi-Input Prompt System ===
Output: Explain AI for beginners in a friendly tone
Output: Explain Python for kids in a fun tone
Output: Explain Deep Learning for engineers in a technical tone


## Task 3: Prompt Variations Engine


In [19]:
# Three PromptTemplates: Teaching, Interview, Storytelling
# Each is stored in a dictionary for modular, reusable access

# Define all three variation templates using a dictionary (modular design)
prompt_variations = {
    "teaching": PromptTemplate(
        input_variables=["topic"],
        template="Explain {topic} clearly step by step"
    ),
    "interview": PromptTemplate(
        input_variables=["topic"],
        template="Ask 3 questions about {topic}"
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic"],
        template="Explain {topic} as a story"
    )
}

def get_variation_prompt(style, topic):
    """Retrieve and format a prompt based on the selected style."""
    if style not in prompt_variations:
        raise ValueError("Style must be one of: teaching, interview, storytelling")
    return prompt_variations[style].format(topic=topic)

# Run all three variations for 'Machine Learning'
topic = "Machine Learning"

print("=== Task 3: Prompt Variations Engine ===")
for style in prompt_variations:
    result = get_variation_prompt(style, topic)
    print(f"[{style.upper()}] → {result}")

=== Task 3: Prompt Variations Engine ===
[TEACHING] → Explain Machine Learning clearly step by step
[INTERVIEW] → Ask 3 questions about Machine Learning
[STORYTELLING] → Explain Machine Learning as a story


## Task 4: ChatPromptTemplate System

In [20]:
# System role → defines behavior | User role → dynamic input
# Supported roles: teacher, interviewer, motivator

# Define system behavior descriptions per role (no hardcoding in function)
role_descriptions = {
    "teacher":     "You are an expert teacher. Explain concepts clearly with examples and step-by-step guidance.",
    "interviewer": "You are a professional interviewer. Ask insightful, thought-provoking questions about the topic.",
    "motivator":   "You are an enthusiastic motivator. Inspire the user to learn and explore the topic with energy."
}

def build_chat_prompt(role, topic):
    """
    Build a ChatPromptTemplate dynamically based on role and topic.
    Returns the formatted message list.
    """
    if role not in role_descriptions:
        raise ValueError("Role must be one of: teacher, interviewer, motivator")

    # System message template – injects role behavior dynamically
    system_template = SystemMessagePromptTemplate.from_template(
        "{role_description}"
    )

    # Human/User message template – dynamic topic input
    human_template = HumanMessagePromptTemplate.from_template(
        "Please help me understand {topic}."
    )

    # Combine into a ChatPromptTemplate
    chat_prompt = ChatPromptTemplate.from_messages([system_template, human_template])

    # Format the prompt with actual values
    formatted = chat_prompt.format_messages(
        role_description=role_descriptions[role],
        topic=topic
    )
    return formatted

# Test: role = 'teacher', topic = 'Neural Networks'
print("=== Task 4: ChatPromptTemplate System ===")

test_roles = [
    {"role": "teacher",     "topic": "Neural Networks"},
    {"role": "interviewer", "topic": "Deep Learning"},
    {"role": "motivator",   "topic": "Artificial Intelligence"},
]

for entry in test_roles:
    messages = build_chat_prompt(**entry)
    print(f"\n[Role: {entry['role'].upper()} | Topic: {entry['topic']}]")
    for msg in messages:
        print(f"  {msg.type.upper()}: {msg.content}")

=== Task 4: ChatPromptTemplate System ===

[Role: TEACHER | Topic: Neural Networks]
  SYSTEM: You are an expert teacher. Explain concepts clearly with examples and step-by-step guidance.
  HUMAN: Please help me understand Neural Networks.

[Role: INTERVIEWER | Topic: Deep Learning]
  SYSTEM: You are a professional interviewer. Ask insightful, thought-provoking questions about the topic.
  HUMAN: Please help me understand Deep Learning.

[Role: MOTIVATOR | Topic: Artificial Intelligence]
  SYSTEM: You are an enthusiastic motivator. Inspire the user to learn and explore the topic with energy.
  HUMAN: Please help me understand Artificial Intelligence.


## Task 5: Input Validation Layer

In [21]:
# audience ∈ [beginner, intermediate, expert]
# tone ∈ [formal, casual, fun]
# Strategy: raise ValueError for invalid inputs

# Define valid options as constants (not hardcoded inside function logic)
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES = ["formal", "casual", "fun"]

# Default fallback values
DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE = "casual"

def validate_inputs(audience, tone, use_defaults=False):

    audience = audience.lower().strip()
    tone = tone.lower().strip()

    # Validate audience
    if audience not in VALID_AUDIENCES:
        if use_defaults:
            print(f"  [WARNING] Invalid audience '{audience}'. Using default: '{DEFAULT_AUDIENCE}'")
            audience = DEFAULT_AUDIENCE
        else:
            raise ValueError(f"Invalid audience '{audience}'. Must be one of: {VALID_AUDIENCES}")

    # Validate tone
    if tone not in VALID_TONES:
        if use_defaults:
            print(f"  [WARNING] Invalid tone '{tone}'. Using default: '{DEFAULT_TONE}'")
            tone = DEFAULT_TONE
        else:
            raise ValueError(f"Invalid tone '{tone}'. Must be one of: {VALID_TONES}")

    return audience, tone


print("=== Task 5: Input Validation Layer ===")

# Test 1: Valid inputs
print("\n[Test 1 - Valid inputs]")
result = validate_inputs("beginner", "fun")
print(f"  Validated: audience={result[0]}, tone={result[1]}")

# Test 2: Invalid inputs – use_defaults=True (assigns fallback values)
print("\n[Test 2 - Invalid inputs with defaults]")
result = validate_inputs("child", "dramatic", use_defaults=True)
print(f"  Validated: audience={result[0]}, tone={result[1]}")

# Test 3: Invalid inputs – raises ValueError
print("\n[Test 3 - Invalid inputs raising ValueError]")
try:
    validate_inputs("child", "dramatic", use_defaults=False)
except ValueError as e:
    print(f"  Caught ValueError: {e}")

=== Task 5: Input Validation Layer ===

[Test 1 - Valid inputs]
  Validated: audience=beginner, tone=fun

[Test 2 - Invalid inputs with defaults]
  [WARNING] Invalid audience 'child'. Using default: 'beginner'
  [WARNING] Invalid tone 'dramatic'. Using default: 'casual'
  Validated: audience=beginner, tone=casual

[Test 3 - Invalid inputs raising ValueError]
  Caught ValueError: Invalid audience 'child'. Must be one of: ['beginner', 'intermediate', 'expert']


## Task 6: Prompt Generator App

In [22]:
# Integrates: validation + style selection + PromptTemplate
# Pipeline: User Input → Validation → Template → Output

# Style-specific PromptTemplates (modular, separate from logic)
style_templates = {
    "teaching": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} teaching style, step by step."
    ),
    "interview": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Generate interview questions about {topic} for {audience} in a {tone} tone."
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} storytelling style."
    )
}

def generate_prompt(topic, audience, tone, style):

    # Step 1: Validate inputs using Task 5's validation function
    audience, tone = validate_inputs(audience, tone, use_defaults=True)

    # Step 2: Validate style
    style = style.lower().strip()
    if style not in style_templates:
        raise ValueError(f"Invalid style '{style}'. Must be one of: {list(style_templates.keys())}")

    # Step 3: Select template and format prompt (logic separated from template)
    selected_template = style_templates[style]
    final_prompt = selected_template.format(topic=topic, audience=audience, tone=tone)

    return final_prompt


print("=== Task 6: Prompt Generator App ===")

# Example from assignment: Neural Networks | beginners | fun | storytelling
example = generate_prompt(
    topic="Neural Networks",
    audience="beginner",
    tone="fun",
    style="storytelling"
)
print(f"Example Output:\n  → {example}")

# Additional test cases
additional_tests = [
    {"topic": "Python",          "audience": "intermediate", "tone": "casual",  "style": "teaching"},
    {"topic": "Data Science",    "audience": "expert",       "tone": "formal",  "style": "interview"},
    {"topic": "Deep Learning",   "audience": "beginner",     "tone": "fun",     "style": "storytelling"},
]

print("\nAdditional Tests:")
for test in additional_tests:
    result = generate_prompt(**test)
    print(f"  → {result}")

=== Task 6: Prompt Generator App ===
Example Output:
  → Explain Neural Networks for beginner in a fun storytelling style.

Additional Tests:
  → Explain Python for intermediate in a casual teaching style, step by step.
  → Generate interview questions about Data Science for expert in a formal tone.
  → Explain Deep Learning for beginner in a fun storytelling style.


## Task 7: Template Reusability Test

In [23]:
# ONE template → 5 different inputs → 5 different outputs
# Goal: Same structure, completely different content

# Single reusable master template
master_template = PromptTemplate(
    input_variables=["topic", "audience", "tone", "style"],
    template=(
        "You are a {style} expert. "
        "Explain {topic} to a {audience}-level learner in a {tone} tone."
    )
)

# 5 completely different input combinations
reusability_inputs = [
    {"topic": "Machine Learning",   "audience": "beginner",     "tone": "fun",     "style": "teacher"},
    {"topic": "Blockchain",         "audience": "intermediate", "tone": "casual",  "style": "journalist"},
    {"topic": "Quantum Computing",  "audience": "expert",       "tone": "formal",  "style": "researcher"},
    {"topic": "Natural Language Processing", "audience": "beginner", "tone": "friendly", "style": "mentor"},
    {"topic": "Reinforcement Learning", "audience": "intermediate", "tone": "technical", "style": "engineer"},
]

print("=== Task 7: Template Reusability Test ===")
print(f"Template Used: '{master_template.template}'\n")
print("--- 5 Different Outputs from the SAME Template ---")

for i, inputs in enumerate(reusability_inputs, start=1):
    # Same template, different data each time
    output = master_template.format(**inputs)
    print(f"\n[Run {i}]")
    print(f"  Input : topic={inputs['topic']}, audience={inputs['audience']}, "
          f"tone={inputs['tone']}, style={inputs['style']}")
    print(f"  Output: {output}")

print("\n✅ Reusability confirmed: Same template structure, 5 unique outputs.")

=== Task 7: Template Reusability Test ===
Template Used: 'You are a {style} expert. Explain {topic} to a {audience}-level learner in a {tone} tone.'

--- 5 Different Outputs from the SAME Template ---

[Run 1]
  Input : topic=Machine Learning, audience=beginner, tone=fun, style=teacher
  Output: You are a teacher expert. Explain Machine Learning to a beginner-level learner in a fun tone.

[Run 2]
  Input : topic=Blockchain, audience=intermediate, tone=casual, style=journalist
  Output: You are a journalist expert. Explain Blockchain to a intermediate-level learner in a casual tone.

[Run 3]
  Input : topic=Quantum Computing, audience=expert, tone=formal, style=researcher
  Output: You are a researcher expert. Explain Quantum Computing to a expert-level learner in a formal tone.

[Run 4]
  Input : topic=Natural Language Processing, audience=beginner, tone=friendly, style=mentor
  Output: You are a mentor expert. Explain Natural Language Processing to a beginner-level learner in a friend

## Full Pipeline Demo

In [24]:
def full_pipeline(topic, audience, tone, style):

    print("=" * 50)
    print("MINI PROMPT ENGINE – Full Pipeline")
    print("=" * 50)
    print(f"[Step 1] User Input   → topic={topic}, audience={audience}, tone={tone}, style={style}")

    # Step 2: Validation
    validated_audience, validated_tone = validate_inputs(audience, tone, use_defaults=True)
    print(f"[Step 2] Validation   → audience={validated_audience}, tone={validated_tone}")

    # Step 3: Template selection + formatting
    final_prompt = generate_prompt(topic, validated_audience, validated_tone, style)
    print(f"[Step 3] Template     → Style '{style}' template selected")

    # Step 4: Output
    print(f"[Step 4] Final Output → {final_prompt}")
    print("=" * 50)
    return final_prompt


# Run the full pipeline
full_pipeline(
    topic="Artificial Intelligence",
    audience="beginner",
    tone="fun",
    style="storytelling"
)

MINI PROMPT ENGINE – Full Pipeline
[Step 1] User Input   → topic=Artificial Intelligence, audience=beginner, tone=fun, style=storytelling
[Step 2] Validation   → audience=beginner, tone=fun
[Step 3] Template     → Style 'storytelling' template selected
[Step 4] Final Output → Explain Artificial Intelligence for beginner in a fun storytelling style.


'Explain Artificial Intelligence for beginner in a fun storytelling style.'